# MemoryArena baseline smoke test on Kaggle

This notebook runs a small `formal_reasoning_math` smoke test across selected MemoryArena memory baselines. It is intentionally small so you can validate API keys, dependencies, and baseline wiring before launching full runs.

Expected Kaggle secrets: `OPENAI_API_KEY` and `OPENAI_BASE_URL`. The LLM model defaults to `cx/gpt-5.4-mini` through an OpenAI-compatible endpoint.

By default this notebook uses a local Hugging Face embedding model for embedding-based baselines, so `/embeddings` API support is not required. Set `EMBEDDING_PROVIDER = "api"` if you want to use an OpenAI-compatible embedding endpoint instead.

In [1]:
# Install runtime dependencies. Optional memory backends are allowed to fail here;
# their corresponding baseline will be marked failed/skipped later.
import subprocess
import sys

BASE_PACKAGES = [
    "openai>=1.0.0",
    "datasets",
    "fastapi",
    "uvicorn",
    "python-dotenv",
    "requests",
    "tiktoken",
    "rank-bm25",
    "pandas",
    "numpy",
    "sentence-transformers",
    "tqdm",
]

OPTIONAL_PACKAGES = [
    "semantic-text-splitter",
    "faiss-cpu",
    "langchain-core",
    "langchain-openai",
    "langchain-graph-retriever",
    "mem0ai",
    "letta-client",
]

def pip_install(packages, required=True):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    except subprocess.CalledProcessError as exc:
        if required:
            raise
        print(f"Optional install failed for {packages}: {exc}")

pip_install(BASE_PACKAGES, required=True)
for package in OPTIONAL_PACKAGES:
    pip_install([package], required=False)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.8/303.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 27.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.9/418.9 kB 11.4 MB/s eta 0:00:00


In [2]:
# Clone or locate the repository.
import os
from pathlib import Path

REPO_URL = "https://github.com/toanthangO20/MemoryArena-Experiment.git"
REPO_BRANCH = "master"

def looks_like_repo(path: Path) -> bool:
    return (path / "README.md").exists() and (path / "agent").exists() and (path / "memory").exists()

cwd = Path.cwd()
if looks_like_repo(cwd):
    REPO_DIR = cwd
else:
    REPO_DIR = Path("/kaggle/working/MemoryArena-Experiment")
    if not REPO_DIR.exists():
        subprocess.check_call([
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ])
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])

os.chdir(REPO_DIR)
for path in [REPO_DIR, REPO_DIR / "src", REPO_DIR / "env" / "env_systems"]:
    value = str(path)
    if value not in sys.path:
        sys.path.insert(0, value)

print("Repo dir:", REPO_DIR)


Cloning into '/kaggle/working/MemoryArena-Experiment'...


Repo dir: /kaggle/working/MemoryArena-Experiment


In [3]:
# Load Kaggle secrets. Outside Kaggle, existing environment variables are used.
try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    for key in ["OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_API_BASE", "EMBEDDING_API_KEY", "EMBEDDING_BASE_URL"]:
        try:
            value = secrets.get_secret(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
except Exception as exc:
    print(f"Kaggle secrets are unavailable in this runtime: {exc}")

if os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_API_BASE"] = os.getenv("OPENAI_BASE_URL")
elif os.getenv("OPENAI_API_BASE"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_API_BASE")
os.environ.setdefault("EMBEDDING_API_KEY", os.getenv("OPENAI_API_KEY", ""))
os.environ.setdefault("EMBEDDING_BASE_URL", os.getenv("OPENAI_BASE_URL", ""))

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing. Add it to Kaggle secrets before running.")
if not os.getenv("OPENAI_BASE_URL"):
    raise RuntimeError("OPENAI_BASE_URL is missing. Add it to Kaggle secrets before running.")
os.environ.setdefault("NGROK_SKIP_BROWSER_WARNING", "true")

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("EMBEDDING_API_KEY loaded:", bool(os.getenv("EMBEDDING_API_KEY")))
print("EMBEDDING_BASE_URL:", os.getenv("EMBEDDING_BASE_URL"))
print("ngrok header enabled:", os.getenv("NGROK_SKIP_BROWSER_WARNING"))


OPENAI_API_KEY loaded: True
OPENAI_BASE_URL: https://splashed-nastily-stopped.ngrok-free.dev/v1
EMBEDDING_API_KEY loaded: True
EMBEDDING_BASE_URL: https://splashed-nastily-stopped.ngrok-free.dev/v1
ngrok header enabled: true


In [4]:
# Baseline and smoke-test controls. Edit this cell first.

MODEL_NAME = "cx/gpt-5.4-mini"
JUDGE_MODEL_NAME = MODEL_NAME
MEMORY_LLM_MODEL = MODEL_NAME

# Use local HF embeddings by default so Kaggle does not need a paid /embeddings API.
# Set EMBEDDING_PROVIDER = "api" to use API_EMBEDDING_MODEL through EMBEDDING_API_KEY/EMBEDDING_BASE_URL.
EMBEDDING_PROVIDER = "local_hf"  # "local_hf" or "api"
LOCAL_HF_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
API_EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_MODEL = LOCAL_HF_EMBEDDING_MODEL if EMBEDDING_PROVIDER == "local_hf" else API_EMBEDDING_MODEL

HF_DATASET = "ZexueHe/memoryarena"
HF_CONFIG = "formal_reasoning_math"
HF_SPLIT = "test"

# Keep this very small for a real smoke test. Increase after the notebook works.
MAX_TASKS = 1
MAX_SUBTASKS_PER_TASK = 2
MAX_COMPLETION_TOKENS = 512
RUN_ENDPOINT_CHECK = True
RUN_EMBEDDING_ENDPOINT_CHECK = True

# The smoke runner uses direct chat completions with max_tokens and ngrok headers.
RUN_JUDGE = True
SKIP_ON_ERROR = True

# Paper-overlap baselines available in this repo. Edit this list to run one or a subset.
BASELINES_TO_RUN = [
    "long_context",
    "text-embedding-3-small",
    "bm25",
    "memorag",
    "graphrag",
    "letta",
    "mem0",
    "mem0-g",
    "reasoningbank",
]

# Optional additions implemented in the codebase but not part of the Table 3 overlap list above:
# BASELINES_TO_RUN = ["mirix", "none"]
# BASELINES_TO_RUN = ["text-embedding-3-small"]

OUTPUT_DIR = Path("/kaggle/working/memoryarena_smoke_outputs")


In [6]:
import importlib
import json
import time
import traceback
import uuid
from typing import Any, Dict, List, Optional

import pandas as pd
from datasets import load_dataset
from openai import OpenAI

OPENAI_DEFAULT_HEADERS = {
    "ngrok-skip-browser-warning": os.getenv("NGROK_SKIP_BROWSER_WARNING", "true"),
}

class CompatCompletions:
    def __init__(self, completions):
        self._completions = completions

    def create(self, *args, **kwargs):
        if "max_completion_tokens" in kwargs and "max_tokens" not in kwargs:
            kwargs["max_tokens"] = kwargs.pop("max_completion_tokens")
        return self._completions.create(*args, **kwargs)

class CompatChat:
    def __init__(self, chat):
        self.completions = CompatCompletions(chat.completions)

class CompatOpenAIClient:
    def __init__(self, client):
        self._client = client
        self.chat = CompatChat(client.chat)
        self.embeddings = client.embeddings

class EmbeddingItem:
    def __init__(self, embedding):
        self.embedding = embedding

class EmbeddingResponse:
    def __init__(self, vectors):
        self.data = [EmbeddingItem(vector) for vector in vectors]

class LocalHFEmbeddingsEndpoint:
    _models = {}

    def _load_model(self, model_name: str):
        if model_name not in self._models:
            from sentence_transformers import SentenceTransformer

            print(f"Loading local HF embedding model: {model_name}")
            self._models[model_name] = SentenceTransformer(model_name)
        return self._models[model_name]

    def create(self, model=None, input=None, **_kwargs):
        model_name = model or LOCAL_HF_EMBEDDING_MODEL
        if isinstance(input, str):
            texts = [input]
        else:
            texts = list(input or [])
        vectors = self._load_model(model_name).encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        return EmbeddingResponse(vectors.astype(float).tolist())

class LocalHFEmbeddingClient:
    def __init__(self):
        self.embeddings = LocalHFEmbeddingsEndpoint()

class LocalLangChainEmbeddings:
    def __init__(self, client):
        self.client = client

    def embed_documents(self, texts):
        response = self.client.embeddings.create(model=EMBEDDING_MODEL, input=list(texts))
        return [item.embedding for item in response.data]

    def embed_query(self, text):
        response = self.client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
        return response.data[0].embedding

def make_openai_client():
    client = OpenAI(
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL") or None,
        default_headers=OPENAI_DEFAULT_HEADERS,
    )
    return CompatOpenAIClient(client)

def make_embedding_client():
    if EMBEDDING_PROVIDER == "local_hf":
        return LocalHFEmbeddingClient()
    client = OpenAI(
        api_key=os.getenv("EMBEDDING_API_KEY") or os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("EMBEDDING_BASE_URL") or os.getenv("OPENAI_BASE_URL") or None,
        default_headers=OPENAI_DEFAULT_HEADERS,
    )
    return CompatOpenAIClient(client)

class MixedOpenAIClient:
    def __init__(self, chat_client, embedding_client):
        self.chat = chat_client.chat
        self.embeddings = embedding_client.embeddings

def make_memory_api_client():
    return MixedOpenAIClient(make_openai_client(), make_embedding_client())

def verify_llm_endpoint() -> None:
    response = make_openai_client().chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": "Trả lời đúng một từ: ok"}],
        stream=False,
        max_tokens=20,
    )
    print("Endpoint check:", (response.choices[0].message.content or "").strip())

if RUN_ENDPOINT_CHECK:
    verify_llm_endpoint()

def verify_embedding_endpoint() -> tuple[bool, Optional[str]]:
    try:
        response = make_embedding_client().embeddings.create(
            model=EMBEDDING_MODEL,
            input=["ok"],
        )
        dim = len(response.data[0].embedding)
        print(f"Embedding backend check: ok, provider={EMBEDDING_PROVIDER}, model={EMBEDDING_MODEL}, dim={dim}")
        return True, None
    except Exception as exc:
        message = f"{type(exc).__name__}: {str(exc)[:300]}"
        print("Embedding backend check failed:", message)
        return False, message

EMBEDDING_AVAILABLE, EMBEDDING_CHECK_ERROR = (
    verify_embedding_endpoint() if RUN_EMBEDDING_ENDPOINT_CHECK else (True, None)
)
EMBEDDING_REQUIRED_BASELINES = {
    "text-embedding-3-small",
    "memorag",
    "graphrag",
    "reasoningbank",
}

def skip_reason_for_baseline(name: str) -> Optional[str]:
    required = {
        "letta": "LETTA_API_KEY",
        "mem0": "MEM0_API_KEY",
        "mem0-g": "MEM0_API_KEY",
        "mirix": "MIRIX_API_KEY",
    }.get(name)
    if required and not os.getenv(required):
        return f"Missing optional secret {required}"
    if name in EMBEDDING_REQUIRED_BASELINES and not EMBEDDING_AVAILABLE:
        return f"Embedding backend unavailable for {EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}: {EMBEDDING_CHECK_ERROR}"
    return None

def build_memory_system(name: str, user_id: str):
    if name == "none":
        return None

    if name == "long_context":
        module = importlib.import_module("memory.memory_systems.long_context")
        return module.LongContextMemorySystem(user_id=user_id)

    if name in {"bm25", "text-embedding-3-small"}:
        module = importlib.import_module("memory.memory_systems.rag")

        class BaseURLRAGMemorySystem(module.RAGMemorySystem):
            def _init_embedding_client(self):
                if self._embedding_client is not None:
                    return
                self._embedding_client = make_embedding_client()

        memory = BaseURLRAGMemorySystem(retrieval_method=name, user_id=user_id)
        memory._embedding_model = EMBEDDING_MODEL
        return memory

    if name == "memorag":
        module = importlib.import_module("memory.memory_systems.memorag")
        return module.MemoRAGMemorySystem(
            user_id=user_id,
            memory_model=MEMORY_LLM_MODEL,
            embedding_model=EMBEDDING_MODEL,
            api_key=os.getenv("OPENAI_API_KEY"),
            api_endpoint=os.getenv("OPENAI_BASE_URL"),
            api_client=make_memory_api_client(),
        )

    if name == "graphrag":
        module = importlib.import_module("memory.memory_systems.langchain_graphrag")

        class BaseURLGraphRAGMemorySystem(module.GraphRAGMemorySystem):
            def _get_vector_store(self, initial_docs=None):
                if self._vector_store is not None:
                    return self._vector_store
                from langchain_core.vectorstores import InMemoryVectorStore

                if EMBEDDING_PROVIDER == "local_hf":
                    self._embeddings = self._embeddings or LocalLangChainEmbeddings(make_embedding_client())
                else:
                    from langchain_openai import OpenAIEmbeddings

                    kwargs = {"api_key": os.getenv("EMBEDDING_API_KEY") or self.api_key, "model": EMBEDDING_MODEL, "default_headers": OPENAI_DEFAULT_HEADERS}
                    if os.getenv("EMBEDDING_BASE_URL") or os.getenv("OPENAI_BASE_URL"):
                        kwargs["base_url"] = os.getenv("EMBEDDING_BASE_URL") or os.getenv("OPENAI_BASE_URL")
                    self._embeddings = self._embeddings or OpenAIEmbeddings(**kwargs)
                if initial_docs:
                    self._vector_store = InMemoryVectorStore.from_documents(
                        documents=initial_docs,
                        embedding=self._embeddings,
                    )
                else:
                    self._vector_store = InMemoryVectorStore(embedding=self._embeddings)
                return self._vector_store

        return BaseURLGraphRAGMemorySystem(user_id=user_id, api_key=os.getenv("OPENAI_API_KEY"))

    if name == "reasoningbank":
        module = importlib.import_module("memory.memory_systems.reasoningbank")

        class BaseURLReasoningBankMemorySystem(module.ReasoningBankMemorySystem):
            def __init__(self, *args, **kwargs):
                super().__init__(*args, **kwargs)
                self.client = make_openai_client()

            def _get_openai_embedding(self, text: str, maxlen: int = 4096):
                import torch

                response = make_embedding_client().embeddings.create(
                    model=EMBEDDING_MODEL,
                    input=[text[:maxlen]],
                )
                return torch.tensor([response.data[0].embedding], dtype=torch.float32)

        return BaseURLReasoningBankMemorySystem(
            user_id=user_id,
            model_name=MEMORY_LLM_MODEL,
            embedding_model=API_EMBEDDING_MODEL if EMBEDDING_PROVIDER == "local_hf" else EMBEDDING_MODEL,
            storage_path=str(OUTPUT_DIR / "reasoningbank_data"),
            embedding_path=str(OUTPUT_DIR / "reasoningbank_data"),
        )

    if name == "letta":
        module = importlib.import_module("memory.memory_systems.letta")
        return module.LettaMemorySystem(user_id=user_id)

    if name in {"mem0", "mem0-g"}:
        module = importlib.import_module("memory.memory_systems.mem0")
        return module.Mem0MemorySystem(user_id=user_id, enable_graph=(name == "mem0-g"))

    if name == "mirix":
        module = importlib.import_module("memory.memory_systems.mirix")
        return module.MirixMemorySystem(user_id=user_id)

    raise ValueError(f"Unsupported baseline: {name}")


Endpoint check: ok
Loading local HF embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding backend check: ok, provider=local_hf, model=sentence-transformers/all-MiniLM-L6-v2, dim=384


In [7]:
def load_smoke_tasks() -> List[Dict[str, Any]]:
    dataset = load_dataset(HF_DATASET, HF_CONFIG, split=HF_SPLIT)
    records = []
    for task_idx in range(min(MAX_TASKS, len(dataset))):
        row = dataset[task_idx]
        questions = list(row.get("questions") or [])[:MAX_SUBTASKS_PER_TASK]
        answers = list(row.get("answers") or [])[:MAX_SUBTASKS_PER_TASK]
        backgrounds_raw = row.get("backgrounds") or []
        if isinstance(backgrounds_raw, list):
            backgrounds = backgrounds_raw[:MAX_SUBTASKS_PER_TASK]
        else:
            backgrounds = [backgrounds_raw] * len(questions)

        records.append({
            "task_idx": task_idx,
            "paper_name": row.get("paper_name") or f"task_{task_idx}",
            "items": list(zip(questions, answers, backgrounds)),
        })
    return records

def build_math_prompt(task: str, background: Any = None) -> str:
    if "### BACKGROUND" in str(task) or "### PROBLEM" in str(task):
        return str(task)
    return f"""### BACKGROUND:
{background if background else "No information provided."}

### PROBLEM:
{task}"""

def chat_once(model: str, messages: List[Dict[str, str]], max_tokens: int) -> str:
    response = make_openai_client().chat.completions.create(
        model=model,
        messages=messages,
        stream=False,
        max_tokens=max_tokens,
    )
    return (response.choices[0].message.content or "").strip()

def run_agent(prompt: str) -> Dict[str, Any]:
    answer = chat_once(
        MODEL_NAME,
        [
            {"role": "system", "content": "You are a careful math solver. Use the provided background and memory context if useful. Return a concise final answer."},
            {"role": "user", "content": prompt},
        ],
        MAX_COMPLETION_TOKENS,
    )
    return {
        "type": "final",
        "answer": answer,
        "input": prompt,
        "tool_trace": [{"tool": "direct_chat", "result": answer}],
        "tool_info": [{"tool": "direct_chat", "result": answer}],
    }

def judge_answer(question: str, answer: Any, ground_truth: Any) -> tuple[Optional[bool], Optional[str]]:
    if not RUN_JUDGE:
        return None, None
    judge_prompt = f"""
Determine whether the candidate answer is mathematically equivalent to the ground truth for the question.

Question: {question}
Candidate answer: {answer}
Ground truth: {ground_truth}

Respond with exactly one word: yes or no.
"""
    verdict = chat_once(
        JUDGE_MODEL_NAME,
        [
            {"role": "system", "content": "You are a strict mathematical equivalence judge."},
            {"role": "user", "content": judge_prompt},
        ],
        20,
    ).lower()
    return ("yes" in verdict), verdict

def fallback_memory_entry(task: str, action: Dict[str, Any], reward: Optional[float]) -> str:
    answer = action.get("answer") if isinstance(action, dict) else str(action)
    lines = [f"## Task: {task}", f"## solution: {answer}"]
    if reward is not None:
        lines.append(f"## Judge: {'CORRECT' if reward else 'INCORRECT'}")
    return "\n".join(lines)

def run_one_baseline(baseline: str, records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    skip_reason = skip_reason_for_baseline(baseline)
    if skip_reason:
        return [{
            "baseline": baseline,
            "status": "skipped",
            "error": skip_reason,
        }]

    rows = []

    for record in records:
        user_id = f"smoke_{baseline}_{record['task_idx']}_{uuid.uuid4().hex[:8]}"
        memory = build_memory_system(baseline, user_id)

        for subtask_idx, (question, ground_truth, background) in enumerate(record["items"]):
            started = time.time()
            status = "ok"
            error = None
            reward = None
            final_answer = None
            memory_context = None

            try:
                query = build_math_prompt(task=question, background=background)
                prompt = memory.wrap_user_prompt(query) if memory is not None else query
                action = run_agent(prompt)
                final_answer = action.get("answer")
                reward, judge_result = judge_answer(question, final_answer, ground_truth)
                memory_context = None
                if "<memory_context>" in prompt and "</memory_context>" in prompt:
                    memory_context = prompt.split("<memory_context>", 1)[1].split("</memory_context>", 1)[0]

                if memory is not None:
                    entry = fallback_memory_entry(question, action, reward if RUN_JUDGE else None)
                    memory.add_chunk(entry)
            except Exception as exc:
                status = "failed"
                error = f"{type(exc).__name__}: {str(exc)[:500]}"
                if not SKIP_ON_ERROR:
                    raise

            rows.append({
                "baseline": baseline,
                "status": status,
                "task_idx": record["task_idx"],
                "paper_name": record["paper_name"],
                "subtask_idx": subtask_idx,
                "is_correct": reward,
                "seconds": round(time.time() - started, 3),
                "memory_chars": len(memory_context or ""),
                "answer_preview": str(final_answer or "")[:240],
                "error": error,
            })
    return rows


In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
records = load_smoke_tasks()
print(f"Loaded {len(records)} task(s) from {HF_DATASET}/{HF_CONFIG}:{HF_SPLIT}")
print("Baselines:", BASELINES_TO_RUN)

all_rows = []
for baseline in BASELINES_TO_RUN:
    print("\n" + "=" * 80)
    print("Running baseline:", baseline)
    print("=" * 80)
    try:
        rows = run_one_baseline(baseline, records)
    except Exception as exc:
        if not SKIP_ON_ERROR:
            raise
        rows = [{
            "baseline": baseline,
            "status": "failed",
            "error": f"{type(exc).__name__}: {str(exc)[:500]}",
        }]
        traceback.print_exc()
    all_rows.extend(rows)

raw_df = pd.DataFrame(all_rows)
raw_path = OUTPUT_DIR / "smoke_results_raw.csv"
raw_df.to_csv(raw_path, index=False)

if "is_correct" in raw_df.columns:
    ok_rows = raw_df[raw_df["status"].eq("ok")].copy()
    if not ok_rows.empty:
        ok_rows["is_correct_numeric"] = ok_rows["is_correct"].astype(float)
        summary_df = (
            ok_rows.groupby("baseline", dropna=False)
            .agg(
                completed_subtasks=("status", "size"),
                avg_correct=("is_correct_numeric", "mean"),
                avg_seconds=("seconds", "mean"),
                avg_memory_chars=("memory_chars", "mean"),
            )
            .reset_index()
        )
    else:
        summary_df = pd.DataFrame(columns=["baseline", "completed_subtasks", "avg_correct", "avg_seconds", "avg_memory_chars"])
else:
    summary_df = pd.DataFrame()

status_df = raw_df.groupby(["baseline", "status"], dropna=False).size().reset_index(name="count")
summary_path = OUTPUT_DIR / "smoke_results_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved raw results to", raw_path)
print("Saved summary to", summary_path)
display(status_df)
display(summary_df)
display(raw_df.head(20))


README.md: 0.00B [00:00, ?B/s]

data.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/40 [00:00<?, ? examples/s]

Loaded 1 task(s) from ZexueHe/memoryarena/formal_reasoning_math:test
Baselines: ['long_context', 'text-embedding-3-small', 'bm25', 'memorag', 'graphrag', 'letta', 'mem0', 'mem0-g', 'reasoningbank']

Running baseline: long_context


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

,baseline,status,count
0,bm25,ok,2
1,graphrag,ok,2
2,letta,skipped,1
3,long_context,ok,2
4,mem0,skipped,1
5,mem0-g,skipped,1
6,memorag,ok,2
7,reasoningbank,ok,2
8,text-embedding-3-small,ok,2


,baseline,completed_subtasks,avg_correct,avg_seconds,avg_memory_chars
0,bm25,2,1.0,8.3965,965.5
1,graphrag,2,1.0,12.4245,1049.0
2,long_context,2,1.0,12.4810,1020.5
3,memorag,2,1.0,14.7225,976.5
4,reasoningbank,2,1.0,21.8525,923.5
5,text-embedding-3-small,2,1.0,9.8785,973.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,baseline,status,task_idx,paper_name,subtask_idx,is_correct,seconds,memory_chars,answer_preview,error
0,long_context,ok,0.0,2503.19064,0.0,True,13.898,6.0,Define\n\[\n\nu: Z_J \longrightarrow X_{\scA}=...,None
1,long_context,ok,0.0,2503.19064,1.0,True,11.064,2035.0,The zero set of the vanishing ideal \(I\) is e...,None
2,text-embedding-3-small,ok,0.0,2503.19064,0.0,True,12.896,6.0,"Define\n\[\n\nu: Z_J \to X_{\scA}=\Hom(\scA,\R...",None
3,text-embedding-3-small,ok,0.0,2503.19064,1.0,True,6.861,1940.0,The zero set of the vanishing ideal \(I\) is e...,None
4,bm25,ok,0.0,2503.19064,0.0,True,9.467,6.0,"Define\n\[\n\nu: Z_J \to X_{\cA}=\Hom(\cA,\R),...",None
5,bm25,ok,0.0,2503.19064,1.0,True,7.326,1925.0,The zero set of \(I\) is exactly \(C\).\n\nInd...,None
6,memorag,ok,0.0,2503.19064,0.0,True,9.136,6.0,"Define\n\[\n\nu:Z_J\to X_\scA=\Hom(\scA,\R),\q...",None
7,memorag,ok,0.0,2503.19064,1.0,True,20.309,1947.0,The zero set of the vanishing ideal \(I\) is հ...,None
8,graphrag,ok,0.0,2503.19064,0.0,True,16.656,6.0,Define\n\[\n\nu:Z_J\to X_\mathcal A=\Hom(\math...,None
9,graphrag,ok,0.0,2503.19064,1.0,True,8.193,2092.0,The zero set of the ideal\n\[\nI=\{f\in C^\inf...,None


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
